In [ ]:
import plotly.express as px
import requests
import pandas as pd
import os

# 1. Fetch municipality GeoJSON from DAWA
geojson = requests.get(
    "https://api.dataforsyningen.dk/kommuner?format=geojson"
).json()

# Each feature has properties: "kode" (4-digit code), "navn" (name)
# Extract names and codes for a base dataframe
municipalities = [
    {
        "Municipality" : f["properties"]["navn"],
        "KommuneCode"  : f["properties"]["kode"],
        "RegionName"   : f["properties"]["regionsnavn"],
        "RegionCode"   : f["properties"]["regionskode"]
    }
    for f in geojson["features"]
]
df_mun = pd.DataFrame(municipalities)

# os.makedirs("../data/geography", exist_ok=True)
# df_mun.to_csv("../data/geography/municipalities.csv", encoding="utf-8", index=False)

In [16]:
r = requests.get("https://api.dataforsyningen.dk/kommuner")
r.raise_for_status()
data_ = []
for item in r.json():
    data_.append([item['kode'], item['navn']])

municipalities_df = pd.DataFrame(data_, columns=["KommuneCode", "Municipality"])
municipalities_df.head()

,KommuneCode,Municipality
0,0101,København
1,0147,Frederiksberg
2,0151,Ballerup
3,0153,Brøndby
4,0155,Dragør


In [17]:
mun_df = pd.read_csv("../data/geography/municipalities.csv", encoding="utf-8")

In [18]:
print(len(municipalities_df), len(mun_df))

99 99


In [ ]:
import requests
import pandas as pd
from io import StringIO

BASE = "https://api.statbank.dk/v1"


# ─── Step 1: Discover exact variable codes (run this first, once) ────────────

def inspect_table(table_id: str) -> None:
    """Print all variables and their codes for any DST table."""
    r = requests.get(f"{BASE}/tableinfo/{table_id}", params={"lang": "en"})
    r.raise_for_status()
    meta = r.json()
    print(f"\nTable : {meta['id']} — {meta['text']}")
    print(f"Unit  : {meta['unit']}\n")
    for var in meta["variables"]:
        print(f"  Variable  : {var['id']}  ({var['text']})")
        print(f"  Elimination: {var.get('elimination', False)}")
        for v in var["values"][:10]:
            print(f"    {v['id']:15}  {v['text']}")
        if len(var["values"]) > 10:
            print(f"    ... {len(var['values'])} values total")
        print()

# inspect_table("FOLK1A")   # ← uncomment and run once to verify all codes


# ─── Step 2: Get municipality codes from DAWA ────────────────────────────────
# DAWA uses zero-padded 4-digit codes ("0101"),
# DST FOLK1A uses plain 3-digit codes ("101").

def get_municipality_map() -> dict[str, str]:
    r = requests.get("https://api.dataforsyningen.dk/kommuner")
    r.raise_for_status()
    return {
        str(int(k["kode"])): k["navn"]   # "0101" → "101": "København"
        for k in r.json()
    }

name_map   = get_municipality_map()   # {"101": "København", "147": "Frederiksberg", ...}
muni_codes = list(name_map.keys())    # 98 codes in DST format


# ─── Step 3: Build the query ─────────────────────────────────────────────────

# Take Q1 (Jan 1st) of each year as the annual snapshot — 2008–2024
q1_periods = [f"{y}K1" for y in range(2008, 2025)]   # 17 periods

# ── Adjust filters here ──────────────────────────────────────────────────────
FILTERS = [
    {
        "code": "OMRÅDE",
        "values": muni_codes,    # all 98 municipalities
        # swap for a subset, e.g.: ["101", "751", "461"]
        # or use ["*"] for every area (includes regions + whole country)
    },
    {
        "code": "KØN",
        "values": ["TOT"],       # TOT = all genders combined
        # swap for ["M", "K"] to get men and women separately
    },
    {
        "code": "ALDER",
        "values": ["IALT"],      # IALT = all ages combined
        # swap for e.g. ["20", "21", "22"] for specific ages
        # or aggregation codes like "sum(20_64=20;21;22;...;64)"
    },
    {
        "code": "CIVILSTAND",
        "values": ["TOT"],       # TOT = all marital statuses
    },
    {
        "code": "Tid",
        "values": q1_periods,    # 2008K1 … 2024K1
        # range notation also works: ">=2008K1<=2024K1" (gives all quarters)
        # latest period only:       "(1)"
    },
]
# ─────────────────────────────────────────────────────────────────────────────


# ─── Step 4: Fetch data ──────────────────────────────────────────────────────

def fetch(table_id: str, variables: list) -> pd.DataFrame:
    payload = {
        "table":     table_id,
        "format":    "BULK",          # streaming — no cell limit
        "lang":      "en",
        "delimiter": "Semicolon",
        "variables": variables,
        "valuePresentation": "Code"   # Force API to return codes instead of text
    }
    r = requests.post(f"{BASE}/data", json=payload)
    r.raise_for_status()
    df = pd.read_csv(StringIO(r.text), sep=";", thousands=".")
    df.columns = df.columns.str.strip().str.upper()
    return df

raw = fetch("FOLK1A", FILTERS)

print("Columns returned:", raw.columns.tolist())
# → ['OMRÅDE', 'KØN', 'ALDER', 'CIVILSTAND', 'TID', 'INDHOLD']
print(raw.head())


# ─── Step 5: Clean and reshape ───────────────────────────────────────────────

df = (
    raw
    .rename(columns={
        "OMRÅDE":  "municipality_code",
        "TID":     "quarter",
        "INDHOLD": "population",
    })
    .assign(
        year              = lambda d: d["quarter"].str[:4].astype(int),
        municipality_name = lambda d: d["municipality_code"].astype(str).map(name_map),
    )
    # Drop the dimension columns we fixed to a single value
    .drop(columns=["KØN", "ALDER", "CIVILSTAND", "quarter"])
    [["municipality_code", "municipality_name", "year", "population"]]
    .sort_values(["municipality_name", "year"])
    .reset_index(drop=True)
)

print(f"\nShape  : {df.shape}")
print(f"Municip: {df['municipality_code'].nunique()}")
print(f"Years  : {sorted(df['year'].unique())}")
print(f"\n{df[df['municipality_name'] == 'København']}")


# ─── Step 6: Export ──────────────────────────────────────────────────────────

df.to_csv("../data/geography/folk1a_population.csv",   index=False, encoding="utf-8-sig")
# df.to_excel("folk1a_population.xlsx", index=False)
# print("\nDone — saved to CSV and Excel.")

In [23]:
def inspect_table(table_id: str) -> dict:
    r = requests.get(f"{BASE}/tableinfo/{table_id}", params={"lang": "en"})
    r.raise_for_status()
    meta = r.json()
    print(f"\nTable : {meta['id']} — {meta['text']}")
    print(f"Unit  : {meta.get('unit', '?')}\n")
    for var in meta["variables"]:
        elim = var.get("elimination", False)
        print(f"  [{var['id']}]  '{var['text']}'  (can be eliminated: {elim})")
        for v in var["values"]:
            print(f"    {v['id']:20}  {v['text']}")
        print()
    

In [36]:
def fetch(table_id: str, variables: list) -> pd.DataFrame:
    payload = {
        "table":     table_id,
        "format":    "BULK",          # streaming — no cell limit
        "lang":      "en",
        "delimiter": "Semicolon",
        "variables": variables,
        "valuePresentation": "Code"   # Force API to return codes instead of text
    }
    r = requests.post(f"{BASE}/data", json=payload)
    r.raise_for_status()
    df = pd.read_csv(StringIO(r.text), sep=";", thousands=".")
    df.columns = df.columns.str.strip().str.upper()
    return df


In [37]:
# Municipality code '411' (Christiansø) is not present in LABY06, so we must exclude it
valid_muni_codes = [c for c in muni_codes if c != '411']

FILTER_LABY06 = [
    {
        "code" : "KOMGRP",
        "values" : valid_muni_codes
    },

    {
        "code": "ALDER",
        "values" : ["00"]
    },
    {
        "code": "Tid",
        "values" : [str(y) for y in range(2008, 2025)]
    }
]

raw = fetch("LABY06", variables=FILTER_LABY06)

print("Columns returned:", raw.columns.tolist())
print(raw.head())

Columns returned: ['KOMGRP', 'ALDER', 'TID', 'INDHOLD']
   KOMGRP  ALDER   TID  INDHOLD
0     461      0  2023   250388
1     479      0  2023   260178
2     480      0  2023   247290
3     482      0  2023   229432
4     492      0  2023   229051


In [ ]:
df = (
    raw
    .rename(columns={
        "KOMGRP":  "municipality_code",
        "TID":     "year",
        "INDHOLD": "average_income",
    })
    .assign(
        municipality_name = lambda d: d["municipality_code"].astype(str).map(name_map),
    )
    # Drop the dimension columns we fixed to a single value
    .drop(columns=["ALDER"])
    [["municipality_code", "municipality_name", "year", "average_income"]]
    .sort_values(["municipality_name", "year"])
    .reset_index(drop=True)
)

print(df.head())

df.to_csv("../data/geography/laby06_average_income.csv", index=False, encoding="utf-8")

   municipality_code municipality_name  year  average_income
0                580          Aabenraa  2008          166290
1                580          Aabenraa  2009          167106
2                580          Aabenraa  2010          181280
3                580          Aabenraa  2011          184117
4                580          Aabenraa  2012          188090


In [40]:
df_pop = pd.read_csv("../data/geography/folk1a_population.csv")
df_inc = pd.read_csv("../data/geography/laby06_average_income.csv")

# Perform a left join to keep all rows from df_pop.
# Missing matches in df_inc (like code '411') will naturally be filled with NaN
df_combined = pd.merge(
    df_pop, 
    df_inc, 
    on=["municipality_code", "municipality_name", "year"], 
    how="left"
)

print("Combined DataFrame shape:", df_combined.shape)
print("\nShowing Christiansø (411) with NaN for average_income:")
print(df_combined[df_combined["municipality_code"] == 411].head())

df_combined.to_csv("../data/geography/statsbank_combined.csv", index=False, encoding="utf-8-sig")

Combined DataFrame shape: (1683, 5)

Showing Christiansø (411) with NaN for average_income:
     municipality_code municipality_name  year  population  average_income
187                411       Christiansø  2008          96             NaN
188                411       Christiansø  2009          96             NaN
189                411       Christiansø  2010         101             NaN
190                411       Christiansø  2011          94             NaN
191                411       Christiansø  2012         103             NaN
